<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/low_memory_read.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pyarrow.csv as pv
import pyarrow.parquet as pq
import pyarrow as pa

input_file = "/content/sample_data/transactions.csv"
output_file = "/content/sample_data/transactions.parquet"

# Read CSV incrementally
reader = pv.open_csv(
    input_file,
    read_options=pv.ReadOptions(
        block_size=64 * 1024 * 1024   # 64 MB chunks
    ),
    parse_options=pv.ParseOptions(
        delimiter=","
    )
)

writer = None

try:
    for batch in reader:
        # batch is a pyarrow.lib.RecordBatch
        # ParquetWriter.write_table expects a pyarrow.lib.Table
        # Convert RecordBatch to Table
        table = pa.Table.from_batches([batch]) # Correct conversion

        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )

        # Write this batch immediately
        writer.write_table(table)

        # table/batch goes out of scope on next iteration

finally:
    if writer is not None:
        writer.close()

# Add a line to list the created file
print(f"\n--- Listing the generated file: {output_file} ---")
!ls -l {output_file}

In [7]:
!ls -l /content/sample_data/transactions.*


-rw-r--r-- 1 root root 38141987 Aug 17 16:12 /content/sample_data/transactions.csv
-rw-r--r-- 1 root root 13753315 Aug 17 16:14 /content/sample_data/transactions.parquet
